# Applied Data Structures for Backend and AI — Hands-On

**Software Engineering · Week 03+**

Offline notebook: production-oriented structures for retrieval, sketches, caches, routing, placement, and storage tradeoffs.

## 0. Setup

In [ ]:
import hashlib, math, random, heapq, bisect
from collections import OrderedDict, Counter, deque
from time import perf_counter
import numpy as np

random.seed(7)
rng = np.random.default_rng(7)

def bench(label, fn):
    start = perf_counter(); result = fn(); ms = (perf_counter() - start) * 1000
    print(f'{label:<32} {ms:7.3f} ms')
    return result

## 1. Bloom filter: empirical false positives vs theory

In [ ]:
class BloomFilter:
    def __init__(self, bits, hashes):
        self.bits, self.hashes = bits, hashes
        self.array = bytearray((bits + 7) // 8)
    def _indexes(self, item):
        d = hashlib.sha256(str(item).encode()).digest()
        h1, h2 = int.from_bytes(d[:8], 'big'), int.from_bytes(d[8:16], 'big') or 1
        for i in range(self.hashes):
            yield (h1 + i * h2) % self.bits
    def add(self, item):
        for idx in self._indexes(item): self.array[idx // 8] |= 1 << (idx % 8)
    def __contains__(self, item):
        return all(self.array[idx // 8] & (1 << (idx % 8)) for idx in self._indexes(item))

n, m, k = 3000, 48000, 7
bf = BloomFilter(m, k)
for i in range(n): bf.add(f'key-{i}')
trials = 12000
fp = sum(1 for i in range(n, n + trials) if f'key-{i}' in bf)
print('empirical', round(fp / trials, 4), 'theoretical', round((1 - math.exp(-k*n/m)) ** k, 4))
print('false negatives', sum(1 for i in range(n) if f'key-{i}' not in bf))

## 2. Heap top-k retrieval vs full sort

In [ ]:
doc_ids = np.array([f'doc-{i}' for i in range(60_000)])
scores = rng.random(len(doc_ids))
K = 10

def heap_topk():
    return heapq.nlargest(K, zip(scores.tolist(), doc_ids.tolist()))

def sort_topk():
    order = np.argsort(scores)[-K:][::-1]
    return [(float(scores[i]), doc_ids[i]) for i in order]

h = bench('heapq.nlargest top-k', heap_topk)
s = bench('numpy full argsort top-k', sort_topk)
print('same winner', h[0][1] == s[0][1], 'best score', round(h[0][0], 6))

## 3. Trie prefix lookup for routes/autocomplete

In [ ]:
class Trie:
    def __init__(self): self.root = {}
    def insert(self, word):
        node = self.root
        for ch in word: node = node.setdefault(ch, {})
        node['$'] = word
    def prefix(self, pref):
        node = self.root
        for ch in pref:
            if ch not in node: return []
            node = node[ch]
        out, stack = [], [node]
        while stack:
            cur = stack.pop()
            if '$' in cur: out.append(cur['$'])
            stack.extend(v for k, v in cur.items() if k != '$')
        return sorted(out)

trie = Trie()
for route in ['/api/v1/files', '/api/v1/feedback', '/api/v2/files', '/admin/users']:
    trie.insert(route)
print(trie.prefix('/api/v1/f'))

## 4. Union-find for near-duplicate clusters

In [ ]:
class DSU:
    def __init__(self, items): self.parent = {x: x for x in items}; self.rank = {x: 0 for x in items}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.rank[ra] < self.rank[rb]: ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]: self.rank[ra] += 1

docs = ['a', 'b', 'c', 'd', 'e']
dsu = DSU(docs)
for a, b, sim in [('a','b',0.94), ('b','c',0.91), ('d','e',0.96)]:
    if sim >= 0.90: dsu.union(a, b)
clusters = {}
for d in docs: clusters.setdefault(dsu.find(d), []).append(d)
print(list(clusters.values()))

## 5. Count-Min Sketch and HyperLogLog-style estimate

In [ ]:
class CountMinSketch:
    def __init__(self, width=64, depth=4): self.w, self.d, self.rows = width, depth, [[0]*width for _ in range(depth)]
    def _idx(self, item, seed): return int.from_bytes(hashlib.blake2b(str(item).encode(), digest_size=8, person=bytes([seed])).digest(), 'big') % self.w
    def add(self, item, count=1):
        for r in range(self.d): self.rows[r][self._idx(item, r)] += count
    def estimate(self, item): return min(self.rows[r][self._idx(item, r)] for r in range(self.d))

events = ['rag']*120 + ['login']*30 + ['export']*7 + [f'rare-{i}' for i in range(50)]
cms = CountMinSketch()
for e in events: cms.add(e)
print('cms estimates', {k: cms.estimate(k) for k in ['rag', 'login', 'export']})

class TinyHLL:
    def __init__(self, p=8): self.p, self.m, self.reg = p, 1 << p, [0] * (1 << p)
    def add(self, item):
        x = int.from_bytes(hashlib.sha1(str(item).encode()).digest()[:8], 'big')
        idx, rest = x & (self.m - 1), x >> self.p
        rank = 1 if rest == 0 else (64 - self.p) - rest.bit_length() + 1
        self.reg[idx] = max(self.reg[idx], rank)
    def count(self):
        alpha = 0.7213 / (1 + 1.079 / self.m)
        raw = alpha * self.m * self.m / sum(2 ** -r for r in self.reg)
        zeros = self.reg.count(0)
        return self.m * math.log(self.m / zeros) if zeros else raw

hll = TinyHLL()
for e in [f'user-{i}' for i in range(1000)] + [f'user-{i}' for i in range(300)]: hll.add(e)
print('hll estimate for 1000 uniques', round(hll.count()))

## 6. Cache eviction: LRU scan pollution and TinyLFU admission idea

In [ ]:
class LRU:
    def __init__(self, cap): self.cap, self.data = cap, OrderedDict()
    def access(self, key):
        hit = key in self.data
        if hit: self.data.move_to_end(key)
        self.data[key] = True
        if len(self.data) > self.cap: self.data.popitem(last=False)
        return hit

lru = LRU(3)
for k in ['A','B','C','A','scan1','scan2','scan3']:
    lru.access(k)
print('LRU after scan', list(lru.data))

freq = Counter({'A': 20, 'B': 15, 'C': 12})
def admit(candidate, victim):
    return freq[candidate] > freq[victim]
print('TinyLFU-style admits one-hit scan over A?', admit('scan4', 'A'))

## 7. Consistent hashing with virtual nodes

In [ ]:
class Ring:
    def __init__(self, nodes, vnodes=20):
        self.ring = []
        for node in nodes:
            for v in range(vnodes):
                h = int(hashlib.md5(f'{node}:{v}'.encode()).hexdigest(), 16)
                self.ring.append((h, node))
        self.ring.sort(); self.points = [h for h, _ in self.ring]
    def get(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        i = bisect.bisect_left(self.points, h) % len(self.ring)
        return self.ring[i][1]

keys = [f'key-{i}' for i in range(1000)]
r3, r4 = Ring(['a','b','c']), Ring(['a','b','c','d'])
moved = sum(r3.get(k) != r4.get(k) for k in keys) / len(keys)
print('fraction moved after adding node', round(moved, 3))

## 8. B-tree vs LSM-tree toy write/read tradeoff

In [ ]:
# Toy model: sorted list approximates in-place ordered index; LSM buffers writes then merges.
btree = []
lsm_mem, lsm_runs = [], []
for x in [5,1,9,2,8,3,7,4,6]:
    bisect.insort(btree, x)
    lsm_mem.append(x)
    if len(lsm_mem) == 3:
        lsm_runs.append(sorted(lsm_mem)); lsm_mem.clear()
print('btree ordered page view', btree)
print('lsm immutable runs', lsm_runs, 'memtable', lsm_mem)
print('read amplification: check runs + memtable; write path: append then compact')

## Exercises
1. Change Bloom `m` and `k`; plot how false-positive rate moves.
2. Increase candidate count and compare heap versus sort for several `k` values.
3. Add weighted nodes to the consistent-hash ring.
4. Extend the cache demo with a real admission window.

## Links
- Literature note: `02 Literature Notes/Software Engineering/Data Structures, Algorithms & Complexity`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 03+ Bloom Filter False Positive Demo`, `.../SE Week 03+ Heap Top K Vector Retriever`
- MOC: `06 Maps of Content/Software Engineering Concepts`